In [20]:
import os
import gc
import csv
import shutil
import numpy as np
import pandas as pd
import pickle
import itertools

from tqdm import tqdm
from sksurv.metrics import concordance_index_censored

import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch import Tensor
from torch.utils.data.dataloader import default_collate
from torch.utils.data import DataLoader, Dataset

In [21]:
dataHum = pd.read_csv('dataset/MultiomicsFinal.csv', index_col='ID')
multiomics = pd.read_csv('dataset_multiomics/Input_MM_ICH_top2000_norm_Scaled.csv', index_col='ID')

In [22]:
def seed_torch(device, seed=42):
    import random
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if device.type == 'cuda':
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def create_bins(data=None, label_col=None, n_bins=5, eps=1e-6):
    if data is None:
        raise ValueError('Dataset in .csv format is required')

    if not label_col:
        label_col = 'OS censored at TPX  months'
    else:
        assert label_col in data.columns

    patients_df = data.copy()
    uncensored_df = patients_df[patients_df['Outcome at last FU'] == 'Dead']

    disc_labels, q_bins = pd.qcut(uncensored_df[label_col], q=n_bins, retbins=True, labels=False)
    q_bins[-1] = data[label_col].max() + eps
    q_bins[0] = data[label_col].min() - eps

    disc_labels, q_bins = pd.cut(patients_df[label_col], bins=q_bins, retbins=True, labels=False, right=False, include_lowest=True)
    patients_df.insert(2, 'time_label', disc_labels.values.astype(int))

    return patients_df


def custom_collate_fn(batch):
    batch = list(filter(lambda x: x is not None, batch))
    return default_collate(batch)

In [23]:
def save_risk_events_times(epoch, filename, mode, train_risk, train_events, train_times, val_risk, val_events, val_times):
    header = ['epoch', 'type', 'risk_scores', 'event_indicators', 'times']

    with open(filename, mode, newline='') as file:
        writer = csv.writer(file)
        writer.writerow(header)

        writer.writerow([epoch, 'train', train_risk.tolist(), train_events.tolist(), train_times.tolist()])
        writer.writerow([epoch, 'val', val_risk.tolist(), val_events.tolist(), val_times.tolist()])


def save_metrics(epoch, filename, mode, train_loss, val_loss, train_cindex, val_cindex):
    header = ['epoch', 'train_loss', 'val_loss', 'train_c_index', 'val_c_index']
    write_header = not os.path.exists(filename) or mode == 'w'

    with open(filename, mode, newline='') as file:
        writer = csv.writer(file)
        if write_header:
            writer.writerow(header)

        writer.writerow([epoch, train_loss, val_loss, train_cindex, val_cindex])

In [24]:
def cox_ph_loss_sorted(log_h: Tensor, events: Tensor, eps: float = 1e-7) -> Tensor:
    if events.dtype is torch.bool:
        events = events.float()
    events = events.view(-1)
    log_h = log_h.view(-1)
    gamma = log_h.max()
    log_cumsum_h = log_h.sub(gamma).exp().cumsum(0).add(eps).log().add(gamma)
    return - log_h.sub(log_cumsum_h).mul(events).sum().div(events.sum())


def cox_ph_loss(log_h: Tensor, durations: Tensor, events: Tensor, eps: float = 1e-7):
    idx = durations.sort(descending=True)[1]
    events = events[idx]
    log_h = log_h[idx]
    return cox_ph_loss_sorted(log_h, events, eps)

In [25]:
class MultiomicsDataset(Dataset):
    def __init__(self, data_dict, rows=15000):
        self.data_dict = data_dict
        self.patient_ids = list(data_dict.keys())
        self.to_remove = []
        self.rows = rows

    def __len__(self):
        return len(self.patient_ids)

    def __getitem__(self, idx):
        patient_id = self.patient_ids[idx]
        patient_data = self.data_dict[patient_id]

        demog_features = torch.tensor(np.array(patient_data['Demog'], dtype=float), dtype=torch.float32)
        clin_features = torch.tensor(np.array(patient_data['Clin'], dtype=float), dtype=torch.float32)
        genomic_features = torch.tensor(np.array(patient_data['Genomic'], dtype=float), dtype=torch.float32)
        transcr_features = torch.tensor(np.array(patient_data['Transcr'], dtype=float), dtype=torch.float32)

        stainings = None
        for key in patient_data['Stainings'].keys():
            if len(patient_data['Stainings'][key]) != 0:
                for image in patient_data['Stainings'][key]:
                    if stainings is None:
                        stainings = torch.load(os.path.join('features_giga', image))
                    else:
                        stainings = torch.cat((stainings, torch.load(os.path.join('features_giga', image))), dim=0)

        if patient_data['Outcomes'][0] == 'Dead' or patient_data['Outcomes'][0] == 'Death':
            outcome = torch.tensor([1.0], dtype=torch.float32)
        else:
            outcome = torch.tensor([0.0], dtype=torch.float32)

        time = torch.tensor(np.array(patient_data['time'], dtype=float), dtype=torch.float32)

        if stainings is not None and stainings.shape[0] > self.rows:
            self.to_remove.append(idx)
            return None

        features_padding = torch.full((self.rows, 1536), -999, dtype=stainings.dtype)
        if stainings is not None:
            features_padding[:stainings.shape[0], :] = stainings

        masking = torch.tensor(stainings.shape[0])

        sample = {
            'patient_id': patient_id,
            'demog': demog_features,
            'clin': clin_features,
            'genomic': genomic_features,
            'transcr': transcr_features,
            'staining': features_padding,
            'masking': masking,
            'outcome': outcome,
            'time': time
        }

        return sample

In [26]:
class MultiOmicsDF(nn.Module):
    def __init__(self, input_size, dropout, surv_nodes=[512, 32], demo_size=2, clin_size=4, geno_size=40, transcr_size=2021,
                 activation_demo=nn.PReLU, activation_clin=nn.PReLU, activation_geno=nn.Tanh, activation_coxnet=nn.LeakyReLU):
        super(MultiOmicsDF, self).__init__()

        self.attention_net = nn.MultiheadAttention(embed_dim=input_size, num_heads=3, dropout=dropout, batch_first=True)

        self.demo_fc = nn.Sequential(
            nn.Linear(demo_size, 64),
            activation_demo(),
            nn.BatchNorm1d(64)
        )
        self.clin_fc = nn.Sequential(
            nn.Linear(clin_size, 64),
            activation_clin(),
            nn.BatchNorm1d(64)
        )
        self.geno_fc = nn.Sequential(
            nn.Linear(geno_size, 64),
            activation_geno(),
            nn.BatchNorm1d(64)
        )
        self.transcr_fc = nn.Sequential(
            nn.Linear(transcr_size, 128),
            nn.SELU(),
            nn.AlphaDropout(dropout),
        )

        self.deep_fusion = nn.Sequential(
            nn.Linear(1536 + 64 + 64 + 64 + 128, 1024),
            nn.ELU(),
            nn.Dropout(dropout),
            nn.Linear(1024, 512),
            nn.ELU(),
            nn.Dropout(dropout)
        )

        self.surv_dims = surv_nodes
        self.surv_layers = nn.ModuleList()
        self.dropout_coxnet = nn.Dropout(dropout)
        self.activation_coxnet = activation_coxnet()

        for i in range(len(self.surv_dims) - 1):
            self.surv_layers.append(nn.Linear(self.surv_dims[i], self.surv_dims[i+1]))
        
        self.final_layer = nn.Linear(self.surv_dims[-1], 1)

    def coxnet(self, x_concat):
        for i, layer in enumerate(self.surv_layers):
            x_concat = layer(x_concat)
            x_concat = self.activation_coxnet(x_concat)
            if i != len(self.surv_layers) - 1:
                x_concat = self.dropout_coxnet(x_concat)
        return x_concat

    def forward(self, x, mask, demog, clin, genomic, transcr):
        attention = []
        for i in range(x.shape[0]):
            A, _ = self.attention_net(x[i, :int(mask[i]), :], x[i, :int(mask[i]), :], x[i, :int(mask[i]), :], need_weights=False)
            attention.append(torch.mean(A, dim=0, keepdim=True))

            del A
            torch.cuda.empty_cache()
        
        x = torch.cat(attention, dim=0)
        del attention

        demog = self.demo_fc(demog)
        clin = self.clin_fc(clin)
        genomic = self.geno_fc(genomic)
        transcr = self.transcr_fc(transcr)

        x = torch.cat([x, demog, clin, genomic, transcr], dim=1)
        x = self.deep_fusion(x)

        risk = self.coxnet(x)

        del x, demog, clin, genomic, transcr
        torch.cuda.empty_cache()

        risk = self.final_layer(risk)
        return torch.exp(risk)

In [27]:
cv_selected = 0
epochs = 10
activation_list = []

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')


if device.type == 'cuda:0':
    torch.cuda.empty_cache()
    gc.collect()
print('Device:', device)

seed_torch(seed=42, device=device)

modifed_data = create_bins(data=dataHum, label_col=None, n_bins=4, eps=1e-6)

dataset_dict = {}
for file in os.listdir('dataset_multiomics'):
    if file.endswith('.pickle'):
        with open(f"dataset_multiomics/{file}", 'rb') as f:
            pickle_data = pickle.load(f)
        
        for key in pickle_data.keys():
            if key not in dataset_dict.keys():
                dataset_dict[key] = pickle_data[key]
                dataset_dict[key]['time'] = modifed_data.loc[key, 'OS censored at TPX  months']


for cv, split in enumerate(os.listdir('splits')):
    if cv == cv_selected and split.endswith('.csv'):
        print(f'\nCross Validation: Fold {cv}')
        split_data = pd.read_csv(f'splits/{split}')
        ids_train = split_data['train']
        ids_val = split_data['val']

        if ids_val.dtype == 'float64':
            ids_val = ids_val.astype('Int64')

        if ids_train.dtype == 'float64':
            ids_train = ids_train.astype('Int64')

        train_dict = {patient_id: dataset_dict[patient_id] for patient_id in ids_train if patient_id in dataset_dict and '_dp' not in patient_id}
        val_dict = {patient_id: dataset_dict[patient_id] for patient_id in ids_val if patient_id in dataset_dict and '_dp' not in patient_id}

        train_dataset = MultiomicsDataset(train_dict, rows=15000)
        val_dataset = MultiomicsDataset(val_dict, rows=15000)

        train_loader = DataLoader(train_dataset, batch_size=10, collate_fn=custom_collate_fn, shuffle=True, drop_last=True)
        val_loader = DataLoader(val_dataset, batch_size=10, collate_fn=custom_collate_fn, shuffle=False)

Device: cuda:0


In [ ]:
dropout_options = [0.2, 0.4, 0.6]
activations = [nn.ReLU, nn.PReLU, nn.LeakyReLU, nn.ELU, nn.SELU, nn.Tanh]

hyperparam_combinations = list(itertools.product(dropout_options, activations, activations, activations, activations))

best_c_index = 0
best_params = None
mode = 'w'

for dropout, act_demo, act_clin, act_geno, act_coxnet in hyperparam_combinations:
    print(f"Testing config: Dropout={dropout}, Demo={act_demo.__name__}, Clin={act_clin.__name__}, Geno={act_geno.__name__}, CoxNet={act_coxnet.__name__}")
    
    model = MultiOmicsDF(input_size=1536, dropout=dropout,
                         activation_demo=act_demo, activation_clin=act_clin,
                         activation_geno=act_geno, activation_coxnet=act_coxnet)
    model.to(device)
    
    optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=0.001)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=0)
    
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        all_risk_scores = []
        all_censorships = []
        all_real_times = []
        
        for train_sample in tqdm(train_loader, desc=f'Epoch {epoch}, train Loader'):
            features, demog, clin, genomic, transcr, real_time, event_indicator, masking = train_sample['staining'], train_sample['demog'], train_sample['clin'], train_sample['genomic'], train_sample['transcr'], train_sample['time'], train_sample['outcome'], train_sample['masking']
            features, demog, clin, genomic, transcr, real_time, event_indicator, masking = features.to(device), demog.to(device), clin.to(device), genomic.to(device), transcr.to(device), real_time.to(device), event_indicator.to(device), masking.to(device)
            optimizer.zero_grad()
    
            risk = model(features, masking, demog, clin, genomic, transcr)
            loss = criterion_cox(risk, real_time, event_indicator) 
    
            all_risk_scores.append(risk.detach().cpu().numpy())
            all_censorships.append(event_indicator.detach().cpu().numpy())
            all_real_times.append(real_time.detach().cpu().numpy())
    
            del risk, event_indicator, real_time
    
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
    
            nn.utils.clip_grad_norm(parameters=model.parameters(), max_norm=10, norm_type=2.0)
            torch.cuda.empty_cache()

        all_risk_scores = np.concatenate(all_risk_scores).reshape(-1,)
        all_censorships = np.concatenate(all_censorships).reshape(-1,)
        all_real_times = np.concatenate(all_real_times).reshape(-1,)
    
        mask = ~np.isnan(all_censorships) & ~np.isnan(all_real_times) & ~np.isnan(all_risk_scores)
    
        filtered_censorships = all_censorships[mask]
        filtered_real_times = all_real_times[mask]
        filtered_risk_scores = all_risk_scores[mask]

        if not (np.isnan(filtered_censorships).any() or np.isnan(filtered_real_times).any() or np.isnan(filtered_risk_scores).any()):
            c_index = concordance_index_censored(
                (filtered_censorships.reshape(-1,)).astype(bool), 
                filtered_real_times, 
                filtered_risk_scores.reshape(-1,), 
                tied_tol=1e-08
            )[0]
        else:
            c_index = 0.5
        
        scheduler.step(total_loss)

        model.eval()
        with torch.no_grad():
            eval_loss = 0
            eval_risk_scores = []
            eval_censorships = []
            eval_real_times = []
    
            for val_sample in tqdm(val_loader, desc=f'Epoch {epoch}, validation Loader'):
                features_val, demog_val, clin_val, genomic_val, transcr_val, real_times_val, event_indicator_val, masking_val = val_sample['staining'], val_sample['demog'], val_sample['clin'], val_sample['genomic'], val_sample['transcr'], val_sample['time'], val_sample['outcome'], val_sample['masking']
                features_val, demog_val, clin_val, genomic_val, transcr_val, real_times_val, event_indicator_val, masking_val = features_val.to(device), demog_val.to(device), clin_val.to(device), genomic_val.to(device), transcr_val.to(device), real_times_val.to(device), event_indicator_val.to(device), masking_val.to(device)
    
                eval_risk = model(features_val, masking_val, demog_val, clin_val, genomic_val, transcr_val)
                loss_cox_test = criterion_cox(eval_risk, real_times_val, event_indicator_val) 
                eval_loss += loss_cox_test
    
                eval_risk_scores.append(eval_risk.detach().cpu().numpy())
                eval_censorships.append(event_indicator_val.detach().cpu().numpy())
                eval_real_times.append(real_times_val.detach().cpu().numpy())
    
                del eval_risk, event_indicator_val, real_times_val
    
                torch.cuda.empty_cache()
    
            eval_final = eval_loss /len(val_loader)
    
            eval_risk_scores = np.concatenate(eval_risk_scores).reshape(-1,)
            eval_censorships = np.concatenate(eval_censorships).reshape(-1,)
            eval_real_times = np.concatenate(eval_real_times).reshape(-1,)
            
            mask = ~np.isnan(eval_censorships) & ~np.isnan(eval_real_times) & ~np.isnan(eval_risk_scores)
    
            filtered_censorships_val = eval_censorships[mask]
            filtered_real_times_val = eval_real_times[mask]
            filtered_risk_scores_val = eval_risk_scores[mask]

            if not (np.isnan(filtered_censorships_val).any() or np.isnan(filtered_real_times_val).any() or np.isnan(filtered_risk_scores_val).any()):
                c_index_val = concordance_index_censored(
                    (filtered_censorships_val.reshape(-1,)).astype(bool), 
                    filtered_real_times_val, 
                    filtered_risk_scores_val.reshape(-1,), 
                    tied_tol=1e-08
                )[0]
            else:
                c_index_val = 0.5
            #c_index_val = concordance_index_censored(filtered_censorships_val.astype(bool), filtered_real_times_val, filtered_risk_scores_val, tied_tol=1e-08)[0]
            print(f'C-Index Train: {c_index:.4f}, C-Index Validation: {c_index_val:.4f}')
        
        
        if c_index_val > best_c_index:
            best_c_index = c_index_val
            best_params = (dropout, act_demo, act_clin, act_geno, act_coxnet)
            torch.save(model.state_dict(), f'best_model.pth')
            with open("configs.csv", mode=mode, newline='') as file_new:
                writer = csv.writer(file_new)
                if mode == 'w':
                    writer.writerow(["Dropout", "Act_Demo", "Act_Clin", "Act_Geno", "Act_CoxNet", "C-Index"])
                
                writer.writerow([dropout, act_demo, act_clin, act_geno, act_coxnet, best_c_index])

            mode = 'a'

print("Best configuration:", best_params, "with c-index:", best_c_index)
with open("best_config.csv", mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(["Dropout", "Act_Demo", "Act_Clin", "Act_Geno", "Act_CoxNet", "C-Index"])
        writer.writerow([dropout, act_demo, act_clin, act_geno, act_coxnet, best_c_index])